# Interval consistency losses: a medium-scale comparison

Trains the `mc` baseline against every objective in `consistency_losses.md` and tracks all of them against the
exact test set during training.

**Everything about the experiment is defined in this notebook, in plain functions with explicit arguments.**
Nothing is hidden behind a config object you would have to edit on disk. The notebook imports only stable
primitives -- `train`, `LossConfig`, the tokenizer, the exact test set scorer -- and every loop, parameter and
plot below is yours to edit in place. (`maze_consistency/experiments.py` has an equivalent packaged version
driving `python run.py cons-sweep`; this notebook deliberately does not use it.)

| config | consistency term |
| --- | --- |
| `mc` | none -- the baseline |
| `mc_local` | mean of the one-step residuals, `delta_t^2` |
| `mc_all` | every interval, raw, via the `2(n+1)/n * Var(c)` shortcut |
| `mc_all_scaled` | the same divided by `s(n) = (n+2)/3` |
| `mc_mixed` | half local, half `all_scaled` |
| `mc_poly_len` | every interval weighted by its length (the O(dn) polynomial extension) |
| `mc_multiscale` | equal weight per power-of-two length, each divided by the length |

**Every config has an identical data loss.** The consistency term runs on its own batch of rollouts tokenized
in both modes and contributes only its own gradient, so a row differs from the baseline by exactly one term.

1. **Runtime -> Change runtime type -> GPU.** Consistency runs cost about 2x the baseline.
2. Set `REPO` and `BRANCH`. For a **private** repo, add a Colab secret `GITHUB_TOKEN` with read access.
3. **Runtime -> Run all.** If an import error already happened this session, **Runtime -> Restart session** first.

Runs go to Drive (`RUNS_DIR`), so re-running after a disconnect skips finished runs and continues.

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}

In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks) and the exact test set from data/canonical/maze.txt. Deterministic.
!python run.py dataset | tail -4
!python run.py testset | tail -1

## 1. The loss configurations

`LossConfig` is a plain frozen dataclass -- every field is visible here and `replace(cfg, field=value)` makes a
variant. Add a row, drop a row, retune a lambda.

Each objective gets its own `lambda_cons`, chosen so they start at roughly equal **gradient pull** on the
shared weights (~10% of the data loss's gradient norm, measured in `consistency_demo.ipynb` section 5). Equal
loss *values* would not be a fair comparison: raw `all` and `poly_len` sit ~100x above the scaled family, so a
shared lambda would hand them ~100x the gradient. These are a starting point, not a tuned optimum.

In [ ]:
from dataclasses import replace, asdict
from maze_consistency.train import LossConfig
import maze_consistency.consistency as C

CONS_BATCH   = 16     # rollouts the consistency term sees per step; each costs two forward passes
LAMBDA_SCALE = 1.0    # multiplies every lambda below at once

# lambda_cons per objective. Keys are the objectives in consistency.ALL.
LAMBDA = {"local": 0.15, "all": 0.011, "all_scaled": 0.14,
          "mixed": 0.15, "poly_len": 0.007, "multiscale": 0.15}

# The comparison. "mc" is the baseline: same data loss, no consistency term.
LOSSES = {"mc": LossConfig(mc=True)}
for name, lam in LAMBDA.items():
    LOSSES[f"mc_{name}"] = LossConfig(mc=True, cons=True, cons_loss=name,
                                      w_cons=lam * LAMBDA_SCALE, cons_batch=CONS_BATCH)

print(f"available objectives: {sorted(C.ALL)}")
for k, lc in LOSSES.items():
    print(f"  {k:<16} cons={lc.cons!s:<5} loss={lc.cons_loss if lc.cons else '-':<12} "
          f"lambda={lc.w_cons if lc.cons else 0:<8g} cons_batch={lc.cons_batch if lc.cons else 0}")

## 2. The sweep

The whole loop, with every parameter as a named argument. Edit the body directly -- it is only a dozen lines
over `train()`, which returns `(params, cfg)` and writes `params.pkl` + `history.json` per run.

`STEPS = 3000` at `BATCH = 32` is the medium-scale setting: on a Colab GPU the baseline is a couple of minutes
and each consistency run roughly twice that, so the full sweep is ~20-30 minutes. Try `steps=500` first.

In [ ]:
import os
import numpy as np
from maze_consistency.dataset import load as load_data
from maze_consistency.dp import compute_ground_truth
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import ModelConfig, MazeTransformer
from maze_consistency.testset import load_testset, stratified_rows, score
from maze_consistency.train import train, load_run, RUNS_DIR, N_HELDOUT

MAZE, DATA = load_data()
TOK = Tokenizer(MAZE)
N_TRAIN = len(DATA["length"]) - N_HELDOUT

# One fixed set of held-out rollouts, shared by every run, so the consistency numbers compare directly.
HELDOUT = np.random.default_rng(0).choice(np.arange(N_TRAIN, N_TRAIN + N_HELDOUT), 64, replace=False)


def run_sweep(losses, seeds=(0,), steps=3000, batch=32, lr=1e-3, d_model=64, n_layers=2, n_heads=4,
              eval_every=250, eval_per_setting=50, heldout=HELDOUT, log_every=100, prefix="cons",
              skip_existing=True, log=print):
    """Train every (loss config, seed) pair into RUNS_DIR/<prefix>/<name>_s<seed>/.

    losses            {run name: LossConfig}
    seeds             one run per seed; the plots below show a min..max band when there are several
    eval_per_setting  exact-test rows scored per setting at each checkpoint (same rows for every run)
    heldout           rollouts the consistency losses are measured on at each checkpoint (same for every run)
    skip_existing     leave finished runs alone, so a disconnected session resumes where it stopped
    """
    ts = load_testset()
    rows = stratified_rows(ts, eval_per_setting, seed=0)
    cfg = ModelConfig.for_tokenizer(TOK, d_model=d_model, n_layers=n_layers, n_heads=n_heads)
    cons_eval = C.make_heldout_eval(MazeTransformer(cfg), TOK, MAZE, DATA, heldout)

    def eval_fn(params, fwd):                      # edit to track whatever you want; keys become the history
        m = score(params, fwd, TOK, ts, rows)      # act_kl / value_kl / start_kl / dyn_nll vs the exact DP
        m.pop("per_setting")
        m.update(cons_eval(params))                # cons/<objective> and diag/*, identical batch every run
        return m

    done = {}
    for name, lc in losses.items():
        for seed in seeds:
            run = f"{prefix}/{name}_s{seed}"
            if skip_existing and os.path.exists(os.path.join(RUNS_DIR, run, "history.json")):
                log(f"[skip] {run} exists")
                continue
            done[run] = train(name=run, steps=steps, batch=batch, lr=lr, d_model=d_model, n_layers=n_layers,
                              n_heads=n_heads, seed=seed, loss=lc, eval_fn=eval_fn, eval_every=eval_every,
                              log_every=log_every, log=log)
    return done


PREFIX = "cons"
run_sweep(LOSSES, seeds=(0,), steps=3000, batch=32, eval_every=250, eval_per_setting=50, prefix=PREFIX)

## 3. Exact-test metrics

`act_kl` is KL(true action distribution || model) against the DP's exact answer, `value_kl` the same for the
value head. `bin 10` / `bin 11` are high-return conditions; `best far` is near-optimal behaviour from far
starts, which the random-walk training data never demonstrates and where a consistency term has the most room
to help. Lower is better. Change `metrics` and `settings` to look at `start_kl`, `dyn_nll`, or other bins.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt


def load_history(prefix, which="test"):
    """{run name: [history per seed]}. which="test" -> exact-test metrics at each checkpoint;
    which="train" -> logged loss parts, including cons / cond_gap / info_gain."""
    root = os.path.join(RUNS_DIR, prefix)
    out = {}
    for d in sorted(os.listdir(root)) if os.path.isdir(root) else []:
        p = os.path.join(root, d, "history.json")
        if os.path.exists(p):
            with open(p) as f:
                out.setdefault(d.rsplit("_s", 1)[0], []).append(json.load(f)[which])
    return out


def plot_curves(prefix, metrics=("act_kl", "value_kl"),
                settings=("NOR", "bin 10", "bin 11", "best far"), logy=True, order=None, figsize=(4.2, 3.4)):
    runs = load_history(prefix, "test")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(len(metrics), len(settings),
                           figsize=(figsize[0] * len(settings), figsize[1] * len(metrics)), squeeze=False)
    for i, metric in enumerate(metrics):
        for j, setting in enumerate(settings):
            a, key = ax[i, j], f"{metric}/{setting}"
            for c, name in enumerate(names):
                hists = runs[name]
                steps = [m["step"] for m in hists[0]]
                ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
                a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
                if len(hists) > 1:
                    a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
            if logy:
                a.set_yscale("log")
            a.set_title(key, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=8)
    fig.tight_layout()
    return fig


plot_curves(PREFIX)
plt.show()

## 4. Collapse diagnostics

Every one of these objectives has the same degenerate global optimum: `v_t == u_t` with `b_t` flat in `t` --
ignore R entirely -- makes every residual exactly zero. A falling `cons` curve is not on its own good news.

- **`cond_gap`** = mean `|v_t - u_t|`: how much conditioning on R changes the model's own trajectory
  likelihood. Decaying toward 0 means the conditioned and unconditioned policies have merged.
- **`info_gain`** = `log q_n(R) - log q_0(R)`: how much the reward head learns about R along a trajectory.
  Decaying toward 0 means the reward head has gone flat.

A run whose `cons` falls while either decays has bought agreement by discarding the reward channel -- its
lambda is too high. Both are logged every step and never optimized.

In [ ]:
def plot_diagnostics(prefix, keys=(("tf", "data: teacher-forced next-token loss", True),
                                   ("cons", "consistency objective", True),
                                   ("cond_gap", "cond_gap: mean |v_t - u_t|   (-> 0 = R ignored)", False),
                                   ("info_gain", "info_gain: log q_n(R) - log q_0(R)   (-> 0 = head flat)", False)),
                     order=None, figsize=(4.6, 3.8)):
    """Train-side view. keys is (history key, panel title, log y). Configs with no consistency term simply
    do not appear in the cons/cond_gap/info_gain panels."""
    runs = load_history(prefix, "train")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(1, len(keys), figsize=(figsize[0] * len(keys), figsize[1]), squeeze=False)
    for j, (key, title, logy) in enumerate(keys):
        a = ax[0, j]
        for c, name in enumerate(names):
            hists = runs[name]
            steps = [m["step"] for m in hists[0]]
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            if np.isnan(ys).all():
                continue
            a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
            if len(hists) > 1:
                a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
        a.set_yscale("log") if logy else a.axhline(0, color="k", lw=.8, ls=":")
        a.set_title(title, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=7)
    fig.tight_layout()
    return fig


plot_diagnostics(PREFIX)
plt.show()

## 5. Held-out consistency, against the exact model

The `cons` value in the training log is whichever objective that run optimizes, on its own moving batch --
not comparable across configs. These are instead **all six losses on one fixed set of held-out rollouts,
identical for every run**, so a row trained on `local` can be read against a row trained on `multiscale`.

They also have an absolute zero. For the true joint distribution every interval residual vanishes exactly:

    v_t - u_t = log piR*(a_t | t, s_t, k) - log(1/4) = log child_h[t,s_t,a_t,k] - log h[t,s_t,k]

and `child_h[t,s,a,k] = h[t+1, next(s,a), k]`, so `delta_t = b_{t+1} - b_t + b_t - b_{t+1} = 0`. The cell
below computes that from the DP as a check -- it comes out at float32 epsilon, ~1e-13. So these curves are
distance from perfect consistency in absolute terms, and 0 is the target, not an estimated floor.

In [ ]:
GT = compute_ground_truth(MAZE)
exact, _ = C.exact_losses(MAZE, GT, DATA, HELDOUT)
print("the true model on the same held-out rollouts (0 = perfectly consistent):")
print("  " + "  ".join(f"{k} {float(np.asarray(v).mean()):.2e}" for k, v in exact.items()))

plot_curves(PREFIX, metrics=("cons",), settings=tuple(C.ALL), figsize=(3.4, 3.2))
plt.show()
plot_curves(PREFIX, metrics=("diag",), settings=("cond_gap", "info_gain", "drift"), logy=False)
plt.show()

## 5. Final numbers

Last checkpoint, averaged over seeds, with the end-of-run diagnostics beside it. `(d)` columns are each config
minus the baseline named in `baseline`, so negative beats baseline.

In [ ]:
def final_table(prefix, baseline="mc", metrics=("act_kl", "value_kl"),
                settings=("NOR", "bin 10", "bin 11", "best far"),
                diag=("cons/local", "cons/all_scaled", "cons/multiscale", "diag/cond_gap", "diag/info_gain"),
                order=None):
    test = load_history(prefix, "test")
    cols = [f"{m}/{s}" for m in metrics for s in settings]
    names = [n for n in (order or LOSSES) if n in test] or list(test)
    mean_last = lambda hs, k: float(np.mean([h[-1].get(k, np.nan) for h in hs]))
    rows = {n: {k: mean_last(test[n], k) for k in cols + list(diag)} for n in names}
    base = rows.get(baseline)
    w = max(len(n) for n in rows) + 2

    print(f"exact-test KL in nats, lower is better. (d) = minus `{baseline}`, so negative beats baseline.")
    for m in metrics:
        group = [f"{m}/{s}" for s in settings]
        print()
        print(" " * w + "".join(f"{k.split('/')[1]:>22}" for k in group))
        print(f"{m:<{w}}" + "".join(f"{'value':>12}{'(d)':>10}" for _ in group))
        for n, r in rows.items():
            line = "".join(f"{r[k]:12.4f}" + ("".rjust(10) if base is None or n == baseline
                                              else f"{r[k] - base[k]:+10.4f}") for k in group)
            print(f"{n:<{w}}" + line)
    print()
    print("held out, same rollouts for every run; the true model scores 0 on every cons/ column")
    print(f"{'config':<{w}}" + "".join(f"{k:>18}" for k in diag))
    for n, r in rows.items():
        print(f"{n:<{w}}" + "".join(f"{r[k]:18.4f}" if np.isfinite(r[k]) else f"{'-':>18}" for k in diag))
    return rows


_ = final_table(PREFIX)

## 7. Is the extra long-range weight earning anything?

`L_all` only beats `L_local` when residuals accumulate coherently instead of cancelling. The cheap test is how
the interval residual grows with interval length: `rms ~ sqrt(l)` is iid diffusion (nothing for the long-range
weighting to catch), `rms ~ l` is coherent drift (the regime `L_all` is built for).

Measured on the trained checkpoints, held out. If the slope sits near 0.5, expect `local` to do about as well
as `all` -- and treat any ranking above as weak evidence.

In [ ]:
def residual_scaling(prefix, names=("mc", "mc_all_scaled"), seed=0, idx=HELDOUT, show_exact=True):
    maze, d, tok = MAZE, DATA, TOK

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ref = None
    for c, name in enumerate(names):
        try:
            params, mcfg = load_run(f"{prefix}/{name}_s{seed}")
        except FileNotFoundError:
            print(f"  (no run for {name}_s{seed})")
            continue
        batch = C.rollout_batch(tok, maze, d, idx)
        t, r, losses, diag = C.evaluate(params, C.make_terms_fn(MazeTransformer(mcfg), tok), batch)
        ell, rms, _ = C.interval_stats_by_length(np.asarray(r["c"]), np.asarray(batch["lengths"]))
        slope = np.polyfit(np.log(ell[1:]), np.log(rms[1:]), 1)[0]
        ref = ref or (ell, rms[0])
        ax[0].loglog(ell, rms, lw=1.6, color=f"C{c}", label=f"{name}  (l^{slope:.2f})")
        ax[1].bar(np.arange(len(C.ALL)) + 0.8 / len(names) * c - 0.4,
                  [float(np.asarray(losses[k]).mean()) for k in C.ALL], 0.8 / len(names),
                  color=f"C{c}", label=name)
        print(f"{name:<16} rms Delta ~ l^{slope:.2f}   cond_gap {float(diag['cond_gap'].mean()):.4f}"
              f"   info_gain {float(diag['info_gain'].mean()):.4f}")
    if show_exact:
        eu, ev, eb = C.exact_terms(maze, GT, d["positions"][idx], d["actions"][idx], d["length"][idx],
                                   maze.outcome_bin(d["length"][idx], d["reached"][idx]))
        er = C.residuals(eu, ev, eb, d["length"][idx])
        e_ell, e_rms, _ = C.interval_stats_by_length(np.asarray(er["c"]), np.asarray(d["length"][idx]))
        ax[0].loglog(e_ell, np.maximum(e_rms, 1e-12), color="k", lw=1.2, label="exact model (~0)")
    if ref:
        ell, r0 = ref
        ax[0].loglog(ell, r0 * np.sqrt(ell), "k:", lw=1, label="~sqrt(l)  iid diffusion")
        ax[0].loglog(ell, r0 * ell, "k--", lw=1, label="~l  coherent drift")
    ax[0].set_xlabel("interval length l"); ax[0].set_ylabel("rms Delta")
    ax[0].set_title("interval residual by length, held out", fontsize=9)
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3, which="both")
    ax[1].set_xticks(range(len(C.ALL))); ax[1].set_xticklabels(list(C.ALL), rotation=40, ha="right", fontsize=8)
    ax[1].set_yscale("log"); ax[1].set_title("every consistency loss, held out", fontsize=9)
    ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis="y")
    fig.tight_layout()
    return fig


residual_scaling(PREFIX, names=("mc", "mc_all_scaled"))
plt.show()

## Hacking this

Everything above is a local function; edit the cell and re-run it.

- **Sweep lambda instead of objective.** Build the dict yourself and use a fresh `prefix`:
  ```python
  L = {f"all_scaled_x{f:g}": replace(LOSSES["mc_all_scaled"], w_cons=0.14 * f) for f in (0.1, 1, 10)}
  run_sweep(L, steps=1500, prefix="lam")
  plot_curves("lam", order=L); final_table("lam", baseline="all_scaled_x1", order=L)
  ```
- **More seeds.** `run_sweep(LOSSES, seeds=(0, 1, 2))`; the plots turn into min..max bands automatically.
- **A different data loss.** The consistency term is independent of `mc`:
  `replace(LOSSES["mc_all_scaled"], mc=False, td=True)` pairs it with the TD value loss, `a=True` adds
  identity (A). `LossConfig` fields print with `asdict(LOSSES["mc"])`.
- **Bigger model.** `run_sweep(LOSSES, d_model=128, n_layers=4, n_heads=8, prefix="big")`.
- **A new objective.** Write a function of the `residuals()` dict, add it to `C.ALL`, give it a `LAMBDA`
  entry, re-run cell 1. It flows through the sweep, the plots and the table with no other change.
- **Track something else.** Edit `eval_fn` inside `run_sweep`; whatever dict it returns becomes the test
  history and is plottable by key via `plot_curves(..., metrics=..., settings=...)`.
- **Resume.** `skip_existing=True` leaves finished runs alone. Delete a run's folder to force a redo.